In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display

# 1. Chargement et préparation
df = pd.read_csv("parkinson.csv")
X = df.drop(columns=['ID', 'Recording', 'Status'])
y = df['Status']
X = pd.get_dummies(X, columns=['Gender'], drop_first=True)

# 2. Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Réduction par PCA
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_scaled)

# 4. Validation croisée
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 5. Modèles
classifiers = {
    "Naïve Bayes": GaussianNB(),
    "SVM (C-SVM)": SVC(kernel='rbf', probability=True, random_state=42),
    "SVM (nu-SVM)": NuSVC(probability=True, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,),
                         max_iter=2000,
                         early_stopping=True,
                         validation_fraction=0.2,
                         random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=6),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# 6. Évaluation avec validation croisée
results_pca = []
for name, clf in classifiers.items():
    print(f"🔍 Évaluation du modèle PCA : {name}")
    y_pred = cross_val_predict(clf, X_pca, y, cv=cv)
    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred)
    recall = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    results_pca.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

    # Matrice de confusion
    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                display_labels=["Sain (0)", "Parkinson (1)"])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(cmap='Blues', ax=ax, colorbar=True)
    plt.title(f"Matrice de Confusion - {name}")
    plt.xlabel("Prédiction")
    plt.ylabel("Réel")
    plt.grid(False)
    plt.tight_layout()
    plt.show()

    # Rapport détaillé
    print(f"\n Rapport de classification - {name} (PCA)")
    print(classification_report(y, y_pred))
    print("-" * 80)

# Résumé des performances
results_df_pca = pd.DataFrame(results_pca).set_index("Model").round(4)
print(" Résultats avec sélection de caractéristiques PCA + Validation Croisée 10 folds")
display(results_df_pca)

# 8. Barplot comparatif
results_df_pca.plot(kind='bar', figsize=(10, 6), colormap='cividis')
plt.title("Performance des Classifieurs (PCA + 10-fold CV)")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.grid(axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
